## How Transformers Solve the Remaining Problems

1. **Sequential computation** — Solved
   Transformers remove recurrence entirely. Instead of processing tokens one-by-one, **self-attention** lets every token attend to every other token in the sequence **simultaneously**. This allows full parallelization across the sequence during training (and mostly during inference too, aside from autoregressive decoding).

2. **Long-range dependencies** — Solved even better
   Every token has a **direct O(1) path** to every other token via self-attention (same idea as attention in Seq2Seq, but now used *within* the encoder and *within* the decoder too, not just encoder→decoder). No positional decay at all.

3. **Fixed-size bottleneck** — Solved
   There's no single compressed vector anywhere — every layer keeps a full set of token representations, and self-attention mixes information across all of them at every layer.

4. **Vanishing gradients over long sequences** — Solved
   Direct connections between all positions + residual connections + layer normalization keep gradients stable across many layers, even for long sequences.

5. **Compute cost per step (attention's tradeoff)** — Mitigated via parallel hardware
   Self-attention is O(n²) in sequence length, same asymptotic cost as attention — **but** because it's fully parallelizable (no sequential dependency), it runs much faster in practice on GPUs/TPUs than RNN-based attention models.

6. **Position information** — New problem, solved with a new trick
   Since there's no recurrence, the model has no inherent sense of token order. Transformers add **positional encodings** (sinusoidal or learned) to the input embeddings to inject sequence order information.

7. **Interpretability** — Improved further
   Multiple attention heads (multi-head attention) let the model attend to different types of relationships (syntax, coreference, position) simultaneously, and each head's attention map can be visualized.

---

## What Transformers Introduce as New Challenges

1. **Quadratic compute/memory cost** — O(n²) attention becomes expensive for very long sequences (this is why later variants like Longformer, Linformer, FlashAttention, etc. exist).

2. **No built-in recurrence/order bias** — Must be manually injected via positional encodings; the model doesn't "naturally" understand sequence order the way RNNs do.

3. **Data/compute hungry** — Transformers generally need larger datasets and more compute to reach strong performance compared to RNNs/LSTMs on small datasets, since they lack RNNs' built-in sequential inductive bias.

---

**Bottom line:** Transformers solve the *sequential bottleneck* by replacing recurrence with parallel self-attention, and solve *long-range dependency* and *fixed-size memory* problems even more thoroughly than attention-augmented RNNs — at the cost of quadratic compute and needing explicit positional information.

---

## Query, Key, and Value 

This is where many people get confused. Don't memorize—understand the analogy.

Imagine you're in a library.

**Query (Q)** = What I'm looking for.

**Key (K)** = Labels on all the books.

**Value (V)**= The actual contents of the books.

You compare your Query with every Key.

The better the match, the more you read that book's Value.

That's exactly what Self-Attention does.

---

## Why Q, K, V (Three Matrices)?

- **Query (Q)** — what this token is *looking for*
- **Key (K)** — what this token *offers* to be matched against
- **Value (V)** — the actual *content* passed forward if matched

### Analogy
Like a library search: **Query** = your search request, **Key** = book labels, **Value** = actual book content. You match Query against Keys, but retrieve Values.

### Why not use one vector for all three?
Each plays a different role:
1. Searching (Q)
2. Being matched (K)
3. Contributing information (V)

Using **separate learned weights** (`W_Q`, `W_K`, `W_V`) lets the model transform the same embedding into three specialized representations, instead of forcing one vector to do all three jobs.

### Computation

Q = x·W_Q, K = x·W_K, V = x·W_V

scores = Q·Kᵀ

weights = softmax(scores)

output = weights·V

### Key benefits
- **Q ≠ K** → allows asymmetric relationships (attention isn't symmetric)
- **V ≠ K** → what matches (K) and what's delivered (V) can differ
- More **expressive power** than reusing a single vector for everything

**In short:** Q, K, V let each token learn separate roles — searching, being found, and contributing content — giving self-attention much more flexibility.

---



# 🚀 DAY 13 – BLOCK 3: TRANSFORMER ARCHITECTURE 

## 🎯 Objectives

* Transformer Encoder
* Transformer Decoder
* Positional Encoding
* Feed Forward Network (FFN)
* Residual Connections
* Layer Normalization
* Encoder–Decoder Attention
* Complete data flow through a Transformer


# Part 1 – High-Level Architecture 

                Input Sentence
                      │
               Token Embeddings
                      │
            Positional Encoding
                      │
        ┌────────────────────────┐
        │   Transformer Encoder  │
        └────────────────────────┘
                      │
              Encoder Outputs
                      │
        ┌────────────────────────┐
        │   Transformer Decoder  │
        └────────────────────────┘
                      │
               Linear Layer
                      │
                 Softmax
                      │
             Predicted Token

# Part 2 – Transformer Encoder 


Input Embeddings
        │
        ▼
Multi-Head Self-Attention
        │
        ▼
Add (Residual)
        │
        ▼
LayerNorm
        │
        ▼
Feed Forward Network
        │
        ▼
Add (Residual)
        │
        ▼
LayerNorm
        │
        ▼
Encoder Output

### What each component does

### 1. Multi-Head Self-Attention

Each word gathers information from every other word.

Output shape remains:

```text
(batch_size, seq_len, d_model)
```

---

### 2. Residual Connection

Instead of using only the attention output:

```text
Output = Attention(X)
```

we do:

```text
Output = X + Attention(X)
```

Why?

* Better gradient flow
* Easier optimization
* Prevents information loss

---

### 3. Layer Normalization

Unlike BatchNorm, LayerNorm normalizes across the features of **each token independently**.

Benefits:

* Stable training
* Faster convergence
* Less sensitive to batch size

---

### 4. Feed Forward Network (FFN)

Every token is processed independently through the same small neural network.

Typically:

```text
Linear
   ↓
ReLU (or GELU)
   ↓
Linear
```

Example dimensions:

```text
512
 ↓
2048
 ↓
512
```

The FFN increases model capacity without mixing information between tokens.

---

# Part 3 – Positional Encoding

Transformers process all words simultaneously.

So how does the model know that:

```text
I love AI
```

is different from:

```text
AI love I
```

It doesn't—unless we provide position information.

---

## Solution

Add a positional vector to every embedding.

```text
Embedding

+

Position Encoding

↓

Final Input
```

The original Transformer paper uses **sinusoidal positional encodings**.

You don't need to memorize the equations today.

Understand the intuition:

* Every position has a unique encoding.
* Nearby positions have similar patterns.
* The model learns word order from these encodings.

---



# Part 4 – Transformer Decoder 

The decoder is slightly more complex because it has **two attention layers**.

```text
Previous Output
       │
Masked Multi-Head Attention
       │
Add & Norm
       │
Encoder-Decoder Attention
       │
Add & Norm
       │
Feed Forward
       │
Add & Norm
       │
Linear
       │
Softmax
       │
Next Word
```

---

## Why Masked Self-Attention?

Suppose you're generating:

```text
I love ___
```

The decoder must **not** look at future words.

During training:

```text
I love AI
```

The word **love** should not see **AI**.

Masking prevents cheating.

---

## Encoder–Decoder Attention

This is different from Self-Attention.

Here:

* **Queries** come from the decoder.
* **Keys** and **Values** come from the encoder outputs.

The decoder asks:

> "Which parts of the input sentence are useful for predicting the next word?"

---

# Part 5 – PyTorch Architecture 

Read the official documentation:

📖 [https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)

Look at the constructor:

```python
nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6
)
```

Understand what each parameter controls.

---





# 🎯 Active Recall

1. Why do Transformers need positional encoding?

2. What is the purpose of residual connections?

       Residual connections add the input of a layer directly to its output. They preserve the original information, improve gradient flow during backpropagation, make very deep networks easier to train, and allow each layer to learn a residual correction instead of an entirely new representation.

3. How is LayerNorm different from BatchNorm?

       BatchNorm: "Normalize across the batch."
       LayerNorm: "Normalize within each sample."
       In a Transformer, LayerNorm normalizes each token independently across its feature dimension (d_model). Each token has its own mean and variance, so normalization does not depend on other tokens in the sentence or on other samples in the batch.

4. What does the Feed Forward Network do?

       Self-Attention allows tokens to exchange information with each other, but it is mostly a weighted combination of token representations. The Feed Forward Network then applies a nonlinear transformation to each token independently, increasing the model's expressive power and enabling it to learn more complex feature representations.

5. Why does the decoder use masked self-attention?

6. What is the difference between Self-Attention and Encoder–Decoder Attention?

7. What are the inputs to the encoder?

8. What are the inputs to the decoder?

---



# Block- 4

**torch.tensor**- Regular torch.Tensor

* Shape: dense, rectangular — (batch, seq_len, embed_dim) (or (seq_len, batch, embed_dim) if batch_first=False).
* Variable-length handling: since a regular tensor must be rectangular, shorter sequences are padded with dummy tokens (usually zeros) up  to the length of the longest sequence in the batch.
* Padding mask: you must separately pass a src_key_padding_mask (shape (batch, seq_len), boolean) telling the layer which positions are padding, so attention doesn't attend to them.
* Compute cost: the model still does full computation over the padded positions (attention, FFN, etc.), even though their output is masked/ignored — this wastes FLOPs when sequence lengths vary a lot.

**nested tensor**- Nested Tensor

* Shape: "ragged" — each sequence in the batch keeps its own actual length, no padding tokens are materialized at all.
* Variable-length handling: built directly via torch.nested.nested_tensor([...]) from a list of differently-sized tensors.
* No mask needed: since there's no padding, you don't pass src_key_padding_mask — there's nothing to mask.
* Compute cost: internally, PyTorch can skip computation on the "padding" that would otherwise exist, so you get real speedups and memory savings for batches with high length variance (this is the main motivation — it powers the "fastpath"/nested_tensor optimizations used in things like efficient inference for NLP/ASR models).
* Caveats: nested tensor support is still evolving (fewer ops supported, some restrictions on which layer configs work — e.g., certain activation functions, norm_first settings, or custom attention implementations may not be supported), and debugging/inspection is less convenient than plain tensors.

---


## writing code 

* first file i have to create is config.py, for that think about what i need
1. Device
2. Vocabulary
3. Transformer
4. Training
5. Paths

* what i learnt new iin this block is that embedding dim and nheads are dependent, embedding dim should always be divisible by nheads, we split then and pass it to each head and after computing all the heads we concat them which gives the original embedding dim but we still multiply it with Wo to mix information across the heads








### There are two kinds of positional encoding.
**A. Original Transformer (Attention Is All You Need)**

The paper uses sinusoidal positional encoding.

These values are computed using mathematical functions (sine and cosine).

Example:

Position 0 → [0.0, 1.0, 0.0, 1.0, ...]
Position 1 → [0.84, 0.54, ...]
...

These are not trainable.

They are fixed once computed.


**B. BERT (and many modern Transformers)**

Instead of sine/cosine, BERT uses:

nn.Embedding(max_position, embed_dim)

These positional vectors are trainable.

During training, the model learns the best representation for each position.


**what should you write in init and forward?**

* __init__()

Needs the information required to build the positional encoding once:

embed_dim
max_seq_length

* forward()

Needs only:

x

because x already tells us the batch size, sequence length, and embedding dimension.



You first create:

```text
(100, 512)
```

Then:

```python
pe = pe.unsqueeze(0)
```

which becomes

```text
(1, 100, 512)
```

Now let's verify why this is useful.

Suppose your embeddings are:

```text
x.shape = (32, 20, 512)
```

Your stored positional encoding is:

```text
pe.shape = (1, 100, 512)
```

During the forward pass, you don't need all 100 positions. Your input has only 20 tokens.

So you'll conceptually do:

```text
pe[:, :20, :]
```

which gives:

```text
(1, 20, 512)
```

Now when you add:

```text
(32, 20, 512)
+
(1, 20, 512)
```

PyTorch broadcasts the first dimension:

```text
1  →  32
```

Result:

```text
(32, 20, 512)
```

Perfect.

---


## A useful rule of thumb

Whenever you create a tensor inside an nn.Module, ask yourself:

### Will gradient descent update it?
* Yes → nn.Parameter
* No, but it's still part of the model → register_buffer
* No, and it's just a temporary value inside forward() → regular local variable


### Q: Why does the original Transformer use register_buffer, while BERT uses nn.Embedding?

**Answer:**

* The original Transformer uses fixed sinusoidal positional encodings that should move with the model but should not be updated during training, so they are stored as a buffer. BERT learns positional embeddings during training, so they are implemented as trainable parameters using nn.Embedding.